# Projeto Final - Priorizacao de Municipios para Acoes de Desenvolvimento

### Curso Introdutorio de Python para Ciencia de Dados

**Dataset real:** Brazilian Cities

**Problema formulado:** uma instituicao publica ou organizacao social precisa decidir quais municipios brasileiros devem ser priorizados em acoes de desenvolvimento humano. Para apoiar essa decisao, vamos combinar indicadores de IDHM, populacao e PIB per capita.

## 1. Perguntas de negocio

1. Quais regioes apresentam maior concentracao de municipios com baixo IDHM?
2. Municipios com maior PIB per capita sempre apresentam maior IDHM?
3. Quais municipios populosos combinam baixo IDHM e menor renda per capita?
4. Como criar uma lista objetiva de municipios prioritarios para intervencao?

## 2. Importacoes

In [ ]:
from pathlib import Path
import io
import warnings

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 115
plt.rcParams['figure.facecolor'] = 'white'

## 3. Dataset real

In [ ]:
def carregar_brazilian_cities(caminho):
    with open(caminho, 'r', encoding='utf-8', newline='') as arquivo:
        bruto = arquivo.read()

    linhas = bruto.split('\r\n')

    def corrigir(linha):
        if linha.startswith('"') and linha.endswith('"'):
            linha = linha[1:-1]
        return linha.replace('""', '"')

    return pd.read_csv(io.StringIO('\n'.join(corrigir(linha) for linha in linhas if linha.strip())))

candidatos = [
    Path('../datasets/brazilian_city.csv'),
    Path('projeto-final/datasets/brazilian_city.csv'),
    Path('../../projeto-final/datasets/brazilian_city.csv'),
    Path('datasets/brazilian_city.csv'),
]
caminho_dataset = next(caminho for caminho in candidatos if caminho.exists())
df = carregar_brazilian_cities(caminho_dataset)

print(f'Linhas: {df.shape[0]:,}')
print(f'Colunas: {df.shape[1]}')
df.head()

## 4. Preparacao dos dados

In [ ]:
df = df.rename(columns={
    'IDHM Ranking 2010': 'IDHM_Ranking',
    'IBGE_CROP_PRODUCTION_$': 'IBGE_CROP_PROD',
    'IBGE_1-4': 'IBGE_1a4',
    'IBGE_5-9': 'IBGE_5a9',
    'IBGE_10-14': 'IBGE_10a14',
    'IBGE_15-59': 'IBGE_15a59',
    'IBGE_60+': 'IBGE_60mais',
})

regioes = {
    'Norte': ['AC', 'AP', 'AM', 'PA', 'RO', 'RR', 'TO'],
    'Nordeste': ['AL', 'BA', 'CE', 'MA', 'PB', 'PE', 'PI', 'RN', 'SE'],
    'Centro-Oeste': ['DF', 'GO', 'MT', 'MS'],
    'Sudeste': ['ES', 'MG', 'RJ', 'SP'],
    'Sul': ['PR', 'RS', 'SC'],
}

df['REGIAO'] = df['STATE'].map({uf: regiao for regiao, ufs in regioes.items() for uf in ufs})
df['POP_FINAL'] = df['IBGE_RES_POP'].where(df['IBGE_RES_POP'] > 0, df['ESTIMATED_POP'])
df['DENSIDADE'] = np.where(df['AREA'] > 0, df['POP_FINAL'] / df['AREA'], np.nan)
df['TIPO'] = df['CAPITAL'].map({1: 'Capital', 0: 'Interior'})
df['IDHM_FAIXA'] = pd.cut(
    df['IDHM'],
    bins=[0, 0.499, 0.599, 0.699, 0.799, 1],
    labels=['Muito baixo', 'Baixo', 'Medio', 'Alto', 'Muito alto'],
)

colunas_projeto = ['CITY', 'STATE', 'REGIAO', 'TIPO', 'IDHM', 'IDHM_FAIXA', 'GDP_CAPITA', 'POP_FINAL', 'AREA', 'DENSIDADE']
dados = df[colunas_projeto].dropna(subset=['IDHM', 'GDP_CAPITA', 'POP_FINAL', 'REGIAO']).copy()
dados = dados[dados['GDP_CAPITA'] > 0]

dados.head()

## 5. Qualidade e visao geral

In [ ]:
qualidade = pd.DataFrame({
    'coluna': dados.columns,
    'nulos': dados.isna().sum().values,
    'nulos_%': (dados.isna().mean().values * 100).round(2),
    'tipo': dados.dtypes.astype(str).values,
})

display(qualidade)
display(dados[['IDHM', 'GDP_CAPITA', 'POP_FINAL', 'DENSIDADE']].describe().T.round(2))

## 6. Visualizacoes e analises

### Pergunta 1 - Quais regioes concentram municipios com baixo IDHM?

In [ ]:
idhm_regiao = dados.groupby(['REGIAO', 'IDHM_FAIXA'], observed=True).size().reset_index(name='municipios')

fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=idhm_regiao, x='REGIAO', y='municipios', hue='IDHM_FAIXA', ax=ax)
ax.set_title('Quantidade de municipios por faixa de IDHM e regiao')
ax.set_xlabel('Regiao')
ax.set_ylabel('Municipios')
ax.legend(title='Faixa de IDHM', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

dados.groupby('REGIAO')['IDHM'].median().sort_values(ascending=False).round(3)

**Insight:** a faixa de IDHM por regiao ajuda a localizar desigualdades territoriais. A prioridade nao deve ser definida apenas pelo numero de municipios, mas tambem pelo tamanho da populacao afetada.

### Pergunta 2 - PIB per capita sempre acompanha IDHM?

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
sns.scatterplot(
    data=dados,
    x='GDP_CAPITA',
    y='IDHM',
    hue='REGIAO',
    size='POP_FINAL',
    sizes=(15, 220),
    alpha=0.6,
    ax=ax,
)
ax.set_xscale('log')
ax.set_title('PIB per capita x IDHM')
ax.set_xlabel('PIB per capita em escala log')
ax.set_ylabel('IDHM')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

dados[['GDP_CAPITA', 'IDHM']].corr().iloc[0, 1].round(3)

**Insight:** PIB per capita e IDHM tem relacao positiva, mas nao perfeita. Isso indica que crescimento economico precisa ser convertido em melhoria social para aparecer no desenvolvimento humano.

### Pergunta 3 - Quais municipios populosos merecem atencao?

In [ ]:
populosos_baixo_idhm = (
    dados.query('POP_FINAL >= 50000')
    .sort_values(['IDHM', 'GDP_CAPITA'], ascending=[True, True])
    .head(15)
)

fig, ax = plt.subplots(figsize=(11, 7))
plot_data = populosos_baixo_idhm.sort_values('IDHM', ascending=True).copy()
plot_data['MUNICIPIO_UF'] = plot_data['CITY'] + ' - ' + plot_data['STATE']
sns.barplot(data=plot_data, y='MUNICIPIO_UF', x='IDHM', hue='REGIAO', dodge=False, ax=ax)
ax.set_title('Municipios populosos com menor IDHM')
ax.set_xlabel('IDHM')
ax.set_ylabel('Municipio')
ax.legend(title='Regiao', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

populosos_baixo_idhm[['CITY', 'STATE', 'REGIAO', 'IDHM', 'GDP_CAPITA', 'POP_FINAL']].round(2)

**Insight:** municipios populosos com baixo IDHM representam maior potencial de impacto, pois uma politica bem direcionada pode beneficiar mais pessoas.

### Pergunta 4 - Ranking de prioridade

In [ ]:
def normalizar(serie):
    minimo = serie.min()
    maximo = serie.max()
    return (serie - minimo) / (maximo - minimo)

ranking = dados.copy()
ranking['baixo_idhm_score'] = 1 - normalizar(ranking['IDHM'])
ranking['populacao_score'] = normalizar(np.log1p(ranking['POP_FINAL']))
ranking['baixo_pib_score'] = 1 - normalizar(np.log1p(ranking['GDP_CAPITA']))

ranking['prioridade'] = (
    0.50 * ranking['baixo_idhm_score'] +
    0.30 * ranking['populacao_score'] +
    0.20 * ranking['baixo_pib_score']
)

top_prioridade = ranking.sort_values('prioridade', ascending=False).head(20)
top_prioridade[['CITY', 'STATE', 'REGIAO', 'IDHM', 'GDP_CAPITA', 'POP_FINAL', 'prioridade']].round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
plot_ranking = top_prioridade.sort_values('prioridade').copy()
plot_ranking['MUNICIPIO_UF'] = plot_ranking['CITY'] + ' - ' + plot_ranking['STATE']
sns.barplot(data=plot_ranking, y='MUNICIPIO_UF', x='prioridade', hue='REGIAO', dodge=False, ax=ax)
ax.set_title('Top 20 municipios prioritarios pelo indice proposto')
ax.set_xlabel('Indice de prioridade')
ax.set_ylabel('Municipio')
ax.legend(title='Regiao', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

**Insight:** o ranking combina tres criterios: menor IDHM, maior populacao e menor PIB per capita. Ele nao substitui uma decisao tecnica completa, mas ajuda a transformar dados em uma lista inicial de priorizacao.

## 7. Conclusoes finais

1. O problema de desenvolvimento municipal exige olhar conjunto para indicadores sociais, economicos e demograficos.
2. Municipios com baixo IDHM e populacao relevante devem receber atencao especial, pois concentram maior potencial de impacto social.
3. PIB per capita esta associado ao IDHM, mas nao e suficiente para explicar sozinho o desenvolvimento humano.
4. O ranking criado e uma ferramenta simples de apoio a decisao, util para selecionar municipios candidatos a uma investigacao mais profunda.

## 8. Recomendacoes

- Validar o ranking com indicadores adicionais de educacao, saude e saneamento.
- Comparar municipios dentro da mesma regiao antes de definir investimentos.
- Atualizar os dados sempre que novas bases oficiais forem disponibilizadas.
- Usar o indice como ponto de partida, nao como decisao final automatica.